# E-Commerce Conversion Funnel & Product Optimization

## Business Objective
Identify where customers drop out of the e-commerce purchase journey, compare conversion across important customer/product segments, and translate the findings into prioritized product recommendations.

**Dataset:** Synthetic e-commerce session-level data created for this portfolio project.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/ecommerce_data.csv")
df.head()


## 1. Data Understanding

Each row represents a product-view session. Funnel events are recorded as binary variables:

- `product_view`: user viewed a product
- `add_to_cart`: user added a product
- `checkout`: user started checkout
- `purchase`: user completed purchase
- `revenue`: revenue generated by the session


In [ ]:
print("Rows, columns:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)


## 2. Overall Funnel

Conversion at each stage is calculated relative to the previous stage. This identifies the biggest customer drop-off point.


In [ ]:
visitors = len(df)
product_views = df["product_view"].sum()
carts = df["add_to_cart"].sum()
checkouts = df["checkout"].sum()
purchases = df["purchase"].sum()

funnel = pd.DataFrame({
    "Stage": ["Visitors / Product Views", "Add to Cart", "Checkout", "Purchase"],
    "Users": [product_views, carts, checkouts, purchases]
})

funnel["Stage Conversion %"] = [
    100,
    carts/product_views*100,
    checkouts/carts*100,
    purchases/checkouts*100
]

funnel["Drop-off %"] = [
    0,
    (1-carts/product_views)*100,
    (1-checkouts/carts)*100,
    (1-purchases/checkouts)*100
]

funnel


In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=funnel, x="Stage", y="Users")
plt.title("E-Commerce Conversion Funnel")
plt.ylabel("Users")
plt.xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("../outputs/funnel.png", dpi=200)
plt.show()


## 3. Conversion by Device

A large difference between mobile and desktop conversion can indicate a product or UX opportunity. The result is a signal for investigation, not proof of the root cause.


In [ ]:
device_analysis = df.groupby("device").agg(
    sessions=("session_id","count"),
    purchases=("purchase","sum"),
    revenue=("revenue","sum")
).reset_index()

device_analysis["conversion_rate_%"] = device_analysis["purchases"] / device_analysis["sessions"] * 100
device_analysis["revenue_per_session"] = device_analysis["revenue"] / device_analysis["sessions"]
device_analysis.sort_values("conversion_rate_%", ascending=False)


In [ ]:
plt.figure(figsize=(7,5))
sns.barplot(data=device_analysis, x="device", y="conversion_rate_%")
plt.title("Purchase Conversion by Device")
plt.ylabel("Conversion Rate (%)")
plt.xlabel("Device")
plt.tight_layout()
plt.savefig("../outputs/device_conversion.png", dpi=200)
plt.show()


## 4. Conversion by Traffic Source

This helps distinguish high-intent and low-intent acquisition channels.


In [ ]:
source_analysis = df.groupby("traffic_source").agg(
    sessions=("session_id","count"),
    purchases=("purchase","sum"),
    revenue=("revenue","sum")
).reset_index()

source_analysis["conversion_rate_%"] = source_analysis["purchases"] / source_analysis["sessions"] * 100
source_analysis["revenue_per_session"] = source_analysis["revenue"] / source_analysis["sessions"]
source_analysis.sort_values("conversion_rate_%", ascending=False)


In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=source_analysis, x="traffic_source", y="conversion_rate_%")
plt.title("Purchase Conversion by Traffic Source")
plt.ylabel("Conversion Rate (%)")
plt.xlabel("Traffic Source")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("../outputs/source_conversion.png", dpi=200)
plt.show()


## 5. Conversion by Product Category


In [ ]:
category_analysis = df.groupby("category").agg(
    sessions=("session_id","count"),
    purchases=("purchase","sum"),
    revenue=("revenue","sum"),
    avg_price=("price","mean")
).reset_index()

category_analysis["conversion_rate_%"] = category_analysis["purchases"] / category_analysis["sessions"] * 100
category_analysis["revenue_per_session"] = category_analysis["revenue"] / category_analysis["sessions"]
category_analysis.sort_values("conversion_rate_%", ascending=False)


In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=category_analysis, x="category", y="conversion_rate_%")
plt.title("Purchase Conversion by Product Category")
plt.ylabel("Conversion Rate (%)")
plt.xlabel("Category")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("../outputs/category_conversion.png", dpi=200)
plt.show()


## 6. Price Segment Analysis

Higher-priced products may require stronger trust signals, financing information, reviews, or comparison tools.


In [ ]:
df["price_segment"] = pd.cut(
    df["price"],
    bins=[0, 1000, 5000, 15000, np.inf],
    labels=["Low (<₹1K)", "Medium (₹1K–₹5K)", "High (₹5K–₹15K)", "Premium (>₹15K)"]
)

price_analysis = df.groupby("price_segment", observed=False).agg(
    sessions=("session_id","count"),
    purchases=("purchase","sum"),
    revenue=("revenue","sum")
).reset_index()

price_analysis["conversion_rate_%"] = price_analysis["purchases"] / price_analysis["sessions"] * 100
price_analysis


## 7. Cart Abandonment

Cart abandonment measures the share of customers who add an item to the cart but do not complete the purchase.

`Cart Abandonment = 1 - Purchases / Add-to-Cart`


In [ ]:
cart_sessions = df["add_to_cart"].sum()
cart_purchases = df["purchase"].sum()

cart_abandonment = (1 - cart_purchases / cart_sessions) * 100

print(f"Cart abandonment rate: {cart_abandonment:.2f}%")


## 8. Product Opportunities

We translate analytical findings into hypotheses and measurable actions.

| Observation | Product hypothesis | Recommendation | KPI |
|---|---|---|---|
| Mobile conversion is lower | Mobile UX may create friction | Simplify mobile checkout | Mobile purchase conversion |
| High cart abandonment | Checkout friction may be significant | Cart recovery + simpler checkout | Cart recovery rate |
| Low-converting categories | Product-page trust/information may be weaker | Improve reviews, images, delivery/returns information | Add-to-cart rate |
| Premium products convert less | High-price purchases need more confidence | Add EMI, warranty and comparison information | Premium conversion |
| Low-converting traffic source | Landing-page intent mismatch | Improve source-specific landing pages | Channel conversion |


## 9. Prioritization Framework

### P0 — Mobile Checkout Optimization
High impact / medium effort.

### P0 — Cart Abandonment Recovery
High impact / low effort.

### P0 — Product Page Optimization
High impact / medium effort.

### P1 — Premium Product Trust Improvements
Medium impact / medium effort.

### P1 — Traffic-Specific Landing Pages
Medium impact / low effort.

## Measurement Plan

For major UX changes, use an A/B test:

**Control:** existing experience  
**Treatment:** redesigned experience

Primary metrics:
- Purchase conversion rate
- Checkout completion rate
- Cart abandonment rate
- Revenue per visitor

Guardrail metrics:
- Average order value
- Refund/cancellation rate
- Customer complaints


## 10. Conclusion

The analysis treats the e-commerce journey as a conversion funnel and identifies the stages and segments that require investigation. The key product-management principle is to move from **data → insight → hypothesis → product action → KPI measurement**, rather than treating a chart as the final output.

**Important:** This portfolio project uses synthetic data. Business impact claims are recommendations/hypotheses and are not presented as measured improvements.
